# The Power Hour: Forecasting Electricity Demand

XGBoost regressor predicting hourly electricity demand for the PJM East region from engineered time series features.

**Dataset:** [Hourly Energy Consumption](https://www.kaggle.com/datasets/robikscube/hourly-energy-consumption) - 145,366 hourly observations from PJM East Interconnection (2002-2018).

**Approach:** Temporal feature engineering (calendar, cyclical, lags, rolling stats) + XGBoost with hyperparameter tuning and SHAP interpretability.

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from sklearn.model_selection import ParameterSampler
import shap
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 30)

In [ ]:
C = {
    "primary":    "#1e293b",
    "secondary":  "#334155",
    "accent":     "#2563eb",
    "risk":       "#dc2626",
    "safe":       "#059669",
    "warn":       "#d97706",
    "light_gray": "#f1f5f9",
    "mid_gray":   "#94a3b8",
    "dark_gray":  "#475569",
}

plt.rcParams.update({
    "figure.facecolor": "#ffffff",
    "axes.facecolor": "#ffffff",
    "axes.edgecolor": "#e2e8f0",
    "axes.labelcolor": C["primary"],
    "axes.titlecolor": C["primary"],
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "axes.grid": True,
    "grid.color": "#f1f5f9",
    "grid.linewidth": 0.8,
    "xtick.color": C["dark_gray"],
    "ytick.color": C["dark_gray"],
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "font.family": "sans-serif",
    "font.sans-serif": ["Segoe UI", "Helvetica Neue", "Arial"],
    "figure.dpi": 100,
    "legend.frameon": False,
    "legend.fontsize": 9,
})

def style_axis(ax, title="", subtitle="", xlabel="", ylabel=""):
    if title:
        ax.set_title(title, fontsize=14, fontweight="bold", color=C["primary"], pad=12, loc="left")
    if subtitle:
        ax.text(0, 1.02, subtitle, transform=ax.transAxes, fontsize=10,
                color=C["dark_gray"], style="italic", va="bottom")
    if xlabel:
        ax.set_xlabel(xlabel, fontsize=11, color=C["secondary"])
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=11, color=C["secondary"])
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#e2e8f0")
    ax.spines["bottom"].set_color("#e2e8f0")
    return ax

def fmt_k(x, _=None):
    if abs(x) >= 1e6:
        return f"{x/1e6:.1f}M"
    if abs(x) >= 1e3:
        return f"{x/1e3:.0f}K"
    return f"{x:.0f}"

print("Style configured.")

In [ ]:
df = pd.read_csv("data/PJME_hourly.csv")
df["Datetime"] = pd.to_datetime(df["Datetime"])
df = df.sort_values("Datetime").reset_index(drop=True)
df = df.set_index("Datetime")

# Handle DST: drop duplicate timestamps, forward-fill single-hour gaps
df = df[~df.index.duplicated(keep="first")]
df = df.asfreq("h")
df["PJME_MW"] = df["PJME_MW"].ffill()

print(f"Loaded: {len(df):,} hourly observations")
print(f"Date range: {df.index.min():%Y-%m-%d} to {df.index.max():%Y-%m-%d}")
print(f"Frequency: {df.index.freq}")
print(f"Missing values: {df['PJME_MW'].isna().sum()}")
print(f"\nDemand stats (MW):")
print(df["PJME_MW"].describe().round(0))

---
## Exploratory Data Analysis

In [ ]:
daily = df["PJME_MW"].resample("D").mean()

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(daily.index, daily.values, lw=0.5, color=C["accent"], alpha=0.6)
rolling_30 = daily.rolling(30).mean()
ax.plot(rolling_30.index, rolling_30.values, lw=2, color=C["primary"], label="30-day rolling mean")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))
ax.legend()
style_axis(ax, title="PJM East Daily Average Demand (2002-2018)",
           subtitle="16 years of hourly electricity consumption",
           xlabel="", ylabel="Demand (MW)")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
df_plot = df.copy()
df_plot["month"] = df_plot.index.month
month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
bp = df_plot.boxplot(column="PJME_MW", by="month", ax=ax, patch_artist=True,
                      boxprops=dict(facecolor=C["accent"], alpha=0.3),
                      medianprops=dict(color=C["risk"], lw=2),
                      flierprops=dict(marker=".", markersize=1, alpha=0.05),
                      showfliers=True)
ax.set_xticklabels(month_names)
ax.set_title("")
plt.suptitle("")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))
style_axis(ax, title="Hourly Demand Distribution by Month",
           subtitle="Summer AC and winter heating drive the highest peaks",
           xlabel="Month", ylabel="Demand (MW)")
plt.tight_layout()
plt.show()

In [ ]:
df_plot["hour"] = df_plot.index.hour
df_plot["is_weekend"] = df_plot.index.dayofweek >= 5

hourly_weekday = df_plot[~df_plot["is_weekend"]].groupby("hour")["PJME_MW"].mean()
hourly_weekend = df_plot[df_plot["is_weekend"]].groupby("hour")["PJME_MW"].mean()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(hourly_weekday.index, hourly_weekday.values, lw=2.5, color=C["accent"],
        marker="o", markersize=4, label="Weekday")
ax.plot(hourly_weekend.index, hourly_weekend.values, lw=2.5, color=C["warn"],
        marker="s", markersize=4, label="Weekend")
ax.set_xticks(range(24))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))
ax.legend()
style_axis(ax, title="Average Hourly Load Curve: Weekday vs Weekend",
           subtitle="Weekdays peak during business hours, weekends are flatter and lower",
           xlabel="Hour of Day", ylabel="Demand (MW)")
plt.tight_layout()
plt.show()

In [ ]:
dow_names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
dow_demand = df.groupby(df.index.dayofweek)["PJME_MW"].mean()

fig, ax = plt.subplots(figsize=(10, 5))
colors_dow = [C["accent"]] * 5 + [C["warn"]] * 2
ax.bar(range(7), dow_demand.values, color=colors_dow, alpha=0.8)
ax.set_xticks(range(7))
ax.set_xticklabels(dow_names)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))
for i, v in enumerate(dow_demand.values):
    ax.text(i, v, f"{v:,.0f}", ha="center", va="bottom", fontsize=9)
style_axis(ax, title="Average Demand by Day of Week",
           subtitle="Amber = weekend. Clear weekday/weekend separation.",
           xlabel="", ylabel="Demand (MW)")
plt.tight_layout()
plt.show()

In [ ]:
heatmap_data = df.copy()
heatmap_data["hour"] = heatmap_data.index.hour
heatmap_data["month"] = heatmap_data.index.month
pivot = heatmap_data.groupby(["hour", "month"])["PJME_MW"].mean().unstack()
pivot.columns = month_names

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(pivot, cmap="YlOrRd", ax=ax, linewidths=0.3, linecolor="#e2e8f0",
            fmt=",.0f", annot=True, annot_kws={"size": 8},
            cbar_kws={"label": "Avg Demand (MW)"})
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
style_axis(ax, title="Average Demand Heatmap: Hour x Month",
           subtitle="Peak demand at summer afternoons (July-Aug, 14:00-18:00) and winter mornings",
           xlabel="Month", ylabel="Hour of Day")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
compare_years = [2005, 2010, 2015]
colors_yoy = [C["accent"], C["safe"], C["warn"]]

for year, color in zip(compare_years, colors_yoy):
    year_data = daily[daily.index.year == year]
    ax.plot(range(len(year_data)), year_data.values, lw=1.5, color=color,
            alpha=0.8, label=str(year))

ax.set_xlabel("Day of Year")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))
ax.legend()
style_axis(ax, title="Year-over-Year Demand Comparison",
           subtitle="Daily average demand for 2005, 2010, 2015 - consistent seasonal shape",
           xlabel="Day of Year", ylabel="Demand (MW)")
plt.tight_layout()
plt.show()

---
## Feature Engineering

All features derived from the datetime index and lagged values of the target. No external data sources - everything the model needs comes from the timestamp and demand history.

In [ ]:
df["hour"] = df.index.hour
df["dayofweek"] = df.index.dayofweek
df["month"] = df.index.month
df["dayofyear"] = df.index.dayofyear
df["weekofyear"] = df.index.isocalendar().week.astype(int)
df["quarter"] = df.index.quarter
df["is_weekend"] = (df.index.dayofweek >= 5).astype(int)

# Cyclical encoding
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
df["dow_sin"] = np.sin(2 * np.pi * df["dayofweek"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["dayofweek"] / 7)

print(f"Calendar + cyclical features added: {df.shape[1] - 1} features")

In [ ]:
df["lag_1h"] = df["PJME_MW"].shift(1)
df["lag_24h"] = df["PJME_MW"].shift(24)
df["lag_48h"] = df["PJME_MW"].shift(48)
df["lag_168h"] = df["PJME_MW"].shift(168)

print("Lag features added: lag_1h, lag_24h, lag_48h, lag_168h")

In [ ]:
df["rolling_24h_mean"] = df["PJME_MW"].shift(1).rolling(24).mean()
df["rolling_24h_std"] = df["PJME_MW"].shift(1).rolling(24).std()
df["rolling_7d_mean"] = df["PJME_MW"].shift(1).rolling(168).mean()
df["rolling_7d_std"] = df["PJME_MW"].shift(1).rolling(168).std()

print("Rolling features added: 24h mean/std, 7d mean/std")

In [ ]:
df["days_since_start"] = (df.index - df.index.min()).days

print(f"\nTotal features: {df.shape[1] - 1}")
print(f"NaN rows (from lags/rolling): {df.isna().any(axis=1).sum():,}")
print(f"\nFeature list:")
print([c for c in df.columns if c != "PJME_MW"])

### Feature Matrix Preview

In [ ]:
print(df.dropna().head(10).to_string())

---
## Train / Validation / Test Split

Temporal split - no data leakage across time:
- **Train:** 2002 through 2016-07-31
- **Validation:** 2016-08-01 through 2017-07-31
- **Test:** 2017-08-01 through 2018-08-03

In [ ]:
df_model = df.dropna().copy()

target = "PJME_MW"
feature_cols = [c for c in df_model.columns if c != target]

train = df_model[df_model.index < "2016-08-01"]
val = df_model[(df_model.index >= "2016-08-01") & (df_model.index < "2017-08-01")]
test = df_model[df_model.index >= "2017-08-01"]

X_train, y_train = train[feature_cols], train[target]
X_val, y_val = val[feature_cols], val[target]
X_test, y_test = test[feature_cols], test[target]

print(f"Train: {len(X_train):,} obs  ({X_train.index.min():%Y-%m-%d} to {X_train.index.max():%Y-%m-%d})")
print(f"Val:   {len(X_val):,} obs  ({X_val.index.min():%Y-%m-%d} to {X_val.index.max():%Y-%m-%d})")
print(f"Test:  {len(X_test):,} obs  ({X_test.index.min():%Y-%m-%d} to {X_test.index.max():%Y-%m-%d})")
print(f"Features: {len(feature_cols)}")